# VIS Scheduling Demo

This notebook mirrors `results/run_vis_rted_tds_fixed.ipynb` using local modules and configurable inputs.


In [ ]:
import os
import csv
from dataclasses import dataclass
import sys
from pathlib import Path
from typing import Dict, Iterable, Sequence

import joblib
import numpy as np
import pandas as pd
import torch

# Limit thread usage to keep things lightweight and avoid OMP warnings.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("KMP_AFFINITY", "disabled")
os.environ.setdefault("KMP_INIT_AT_FORK", "FALSE")

REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from models.models import create_model

try:
    from scheduling.virtual_inertia_scheduling import (
        PriceCoefficients,
        compute_dispatch_cost,
        _candidate_generator,
    )
except Exception:
    @dataclass
    class PriceCoefficients:
        inertia: np.ndarray
        damping: np.ndarray
        delta_p: np.ndarray

    def compute_dispatch_cost(*args, **kwargs):
        raise NotImplementedError("compute_dispatch_cost is missing. Fill in scheduling logic.")

    def _candidate_generator(*args, **kwargs):
        raise NotImplementedError("_candidate_generator is missing. Fill in scheduling logic.")


In [ ]:
CASE_PATH = Path("/path/to/ieee39_full_ibrs.xlsx")
TRAIN_CSV = Path("/path/to/training.csv")
MODEL_DIR = Path("/path/to/model_dir")
MODEL_TYPE = "MTLSH"

LOAD_PROFILE_CSV = Path("load.csv")
LOAD_PROFILE_COL = "load"

OUTPUT_DIR = Path("results/vis_scheduling")
TEXT_LOG_NAME = "vis_rted_tds_output.txt"
CSV_NAME = "vis_rted_tds_results.csv"

SYSTEM_FREQ = 50.0
SEED = 42
STEP_DURATION = 10.0
TDS_TSTEP = 0.1
N_ITERS = None
BASE_LOAD_SCALE = 1.0
SKIP_STEPS_PER_ITER = None

FREQ_LIMIT = 0.1
PENALTY_WEIGHT = 1e3
M_BOUNDS = (0.0, 8.0)
D_BOUNDS = (0.0, 4.0)
N_CANDIDATES = 256

TARGET_COLS = None
TARGET_PREFIXES = ("rocof_", "dev", "Delta_P_IBR_")
DROP_COLS = ["sim_id", "seed", "success", "load_step_time", "time_max_dev", "plotter_csv"]

FREQ_LABEL_MAP = {
    "rocof_max": "rocof_max_COI",
    "rocof_min": "rocof_min_COI",
    "dev_down": "devdown_COI",
    "dev_up": "devup_COI",
}
FREQ_METRIC_LABELS = ["devdown_COI", "devup_COI"]
PLOT_TARGETS = None

if not CASE_PATH.exists():
    raise FileNotFoundError(f"Missing case file: {CASE_PATH}")
if not TRAIN_CSV.exists():
    raise FileNotFoundError(f"Missing training CSV: {TRAIN_CSV}")
if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Missing model directory: {MODEL_DIR}")

df_cols = pd.read_csv(TRAIN_CSV, nrows=0).columns.tolist()

if TARGET_COLS is None:
    TARGET_COLS = [
        c for c in df_cols
        if any(c.startswith(prefix) for prefix in TARGET_PREFIXES)
    ]

if not TARGET_COLS:
    raise ValueError("TARGET_COLS is empty. Set TARGET_COLS or TARGET_PREFIXES.")

TARGET_ORDER = list(TARGET_COLS)

drop_set = set(DROP_COLS)
target_set = set(TARGET_ORDER)
FEATURE_ORDER = [c for c in df_cols if c not in target_set and c not in drop_set]

if not FEATURE_ORDER:
    raise ValueError("No feature columns left after applying TARGET_COLS and DROP_COLS.")

def _sort_indexed(cols, prefix):
    return sorted(cols, key=lambda x: int(x[len(prefix):]))

M_COLS = _sort_indexed([c for c in FEATURE_ORDER if c.startswith("M_") and c[2:].isdigit()], "M_")
D_COLS = _sort_indexed([c for c in FEATURE_ORDER if c.startswith("D_") and c[2:].isdigit()], "D_")
IBR_LABELS = _sort_indexed([c for c in TARGET_ORDER if c.startswith("Delta_P_IBR_")], "Delta_P_IBR_")
N_IBR = max(len(M_COLS), len(D_COLS), len(IBR_LABELS))

ROCOF_LABELS = [label for label in TARGET_ORDER if "rocof" in label]

if SKIP_STEPS_PER_ITER is None:
    SKIP_STEPS_PER_ITER = max(0, int(round(float(STEP_DURATION) / float(TDS_TSTEP))))


In [ ]:
def build_feature_vector(features: Dict[str, float]) -> np.ndarray:
    return np.array(
        [float(features.get(key, 0.0)) for key in FEATURE_ORDER],
        dtype=np.float32,
    ).reshape(1, -1)


def build_feature_vector_for_md(
    base_features: Dict[str, float],
    M_vec: Sequence[float],
    D_vec: Sequence[float],
) -> np.ndarray:
    features = dict(base_features)
    for i, (Mi, Di) in enumerate(zip(M_vec, D_vec), start=1):
        features[f"M_{i}"] = float(Mi)
        features[f"D_{i}"] = float(Di)
    return build_feature_vector(features)


def load_trained_model(model_dir: Path):
    x_scaler = joblib.load(model_dir / "x_scaler.pkl")
    y_scaler = joblib.load(model_dir / "y_scaler.pkl")

    model, device = create_model(
        MODEL_TYPE,
        in_dim=len(FEATURE_ORDER),
        out_dim=len(TARGET_ORDER),
    )
    state_path = model_dir / "vis_mlp_state_dict.pt"
    state_dict = torch.load(state_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    return model, x_scaler, y_scaler, device


In [ ]:
import andes

andes.config_logger(stream_level=20)


def add_measurement_devices(ss):
    """Add BusROCOF at every bus."""
    for bus in ss.Bus.as_df().idx.values:
        ss.add(
            model="BusROCOF",
            idx=f"BusROCOF_{bus}",
            name=f"BusROCOF {bus}",
            param_dict=dict(bus=bus, Tr=0.02, Tw=0.1, Tf=0.02),
        )

    existing = list(ss.PMU.as_df().bus.values) if ss.PMU.n > 0 else []
    for bus in ss.Bus.as_df().idx.values:
        if bus not in existing:
            ss.add(model="PMU", param_dict=dict(bus=bus))


model, x_scaler, y_scaler, device = load_trained_model(MODEL_DIR)

ss = andes.load(str(CASE_PATH), setup=False)
add_measurement_devices(ss)
ss.config.freq = float(SYSTEM_FREQ)

pq_base_p = {n: float(v) for n, v in zip(ss.PQ.name.v, ss.PQ.p0.v)} if ss.PQ.n else {}
pq_base_q = {n: float(v) for n, v in zip(ss.PQ.name.v, ss.PQ.q0.v)} if ss.PQ.n else {}
pq_names = list(ss.PQ.name.v) if ss.PQ.n else []

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
text_log_path = OUTPUT_DIR / TEXT_LOG_NAME
csv_path = OUTPUT_DIR / CSV_NAME

csv_fields = [
    "iter",
    "load_scale",
    "success",
    "feasible",
    "cost",
    "score",
    "freq_metric",
]

csv_fields.extend([f"M_{i}" for i in range(1, N_IBR + 1)])
csv_fields.extend([f"D_{i}" for i in range(1, N_IBR + 1)])
for name in TARGET_ORDER:
    csv_fields.append(f"pred_{name}")
    csv_fields.append(f"true_{name}")


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120


In [ ]:
ss.config.freq = float(SYSTEM_FREQ)
ss.name = "IEEE 39-bus"

ss.PQ.config.p2p = 1
ss.PQ.config.q2q = 1
ss.PQ.config.p2z = 0
ss.PQ.config.q2z = 0
ss.PQ.config.p2i = 0
ss.PQ.config.q2i = 0
ss.PQ.config.pq2z = 0

ss.setup()
ss.PFlow.run()

ss.TDS.config.no_tqdm = True
ss.TDS.config.shrinkt = 1
ss.TDS.config.tol = 1e-3
ss.TDS.config.fixt = 0
ss.TDS.config.tstep = float(TDS_TSTEP)
ss.TDS.config.method = "backeuler"
ss.TDS.config.honest = 1
ss.TDS.config.max_iter = 35
ss.TDS.init()

rng = np.random.default_rng(SEED)
regcv1_sn = np.asarray(ss.REGCV1.Sn.v, dtype=float)
genrou_M = np.asarray(ss.GENROU.M.v, dtype=float)
genrou_D = np.asarray(ss.GENROU.D.v, dtype=float)

# M/D features are in % of Sn to match the trained scaler.
current_M = np.asarray(ss.REGCV1.M.v, dtype=float) * 100.0 / regcv1_sn
current_D = np.asarray(ss.REGCV1.D.v, dtype=float) * 100.0 / regcv1_sn


In [ ]:
def update_regcv1_params(ss, M_vec, D_vec) -> None:
    # M_vec/D_vec are in % of device base (Sn).
    for k, (Sn, Mi, Di) in enumerate(zip(ss.REGCV1.Sn.v, M_vec, D_vec)):
        ss.REGCV1.M.v[k] = float(Mi) * float(Sn) / 100.0
        ss.REGCV1.D.v[k] = float(Di) * float(Sn) / 100.0


def compute_md_agg(genrou_M, genrou_D, regcv1_sn, M_vec, D_vec) -> tuple[float, float]:
    M_actual = np.asarray(M_vec, dtype=float) * regcv1_sn / 100.0
    D_actual = np.asarray(D_vec, dtype=float) * regcv1_sn / 100.0
    M_agg = float(np.mean(np.concatenate([genrou_M, M_actual])))
    D_agg = float(np.mean(np.concatenate([genrou_D, D_actual])))
    return M_agg, D_agg


In [ ]:
load_profile_df = pd.read_csv(LOAD_PROFILE_CSV)
if LOAD_PROFILE_COL not in load_profile_df.columns:
    raise ValueError(f"Missing column '{LOAD_PROFILE_COL}' in {LOAD_PROFILE_CSV}")


def apply_load_scale(ss, pq_base_p, pq_base_q, row_idx) -> float:
    load_scale = float(load_profile_df.loc[row_idx, LOAD_PROFILE_COL])
    for uid, name in enumerate(ss.PQ.name.v):
        base_p = pq_base_p.get(name)
        base_q = pq_base_q.get(name)
        if base_p is None or base_q is None:
            continue
        p = float(base_p) * load_scale
        q = float(base_q) * load_scale
        ss.PQ.p0.v[uid] = p
        ss.PQ.q0.v[uid] = q
        ss.PQ.Ppf.v[uid] = p
        ss.PQ.Qpf.v[uid] = q

    return float(load_scale)


In [ ]:
def compute_freq_metrics(t, f, f0=50.0, r=None, tol_hz=0.01):
    """Compute basic frequency and ROCOF metrics for a single trajectory f(t)."""
    t = np.asarray(t, dtype=float)
    f = np.asarray(f, dtype=float)

    if t.size == 0 or f.size == 0:
        return {}

    if t.size != f.size:
        raise ValueError("Time and frequency arrays must have the same length.")

    if r is None:
        if t.size > 1:
            rocof = np.gradient(f, t, edge_order=2 if t.size > 2 else 1)
        else:
            rocof = np.zeros_like(f)
    else:
        rocof = np.asarray(r, dtype=float)
        if rocof.size != f.size:
            raise ValueError("ROCOF array must have the same length as frequency array.")

    tail_len = max(10, t.size // 10)
    f_ss = np.mean(f[-tail_len:])

    idx_min = np.argmin(f)
    idx_max = np.argmax(f)
    f_min = f[idx_min]
    f_max = f[idx_max]

    dev_down = f0 - f_min
    dev_up = f_max - f0
    max_abs_dev = max(abs(f_min - f0), abs(f_max - f0))

    f_after_nadir = f[idx_min:]
    overshoot_up = max(0.0, np.max(f_after_nadir) - f_ss) if f_after_nadir.size > 0 else 0.0

    within_band = np.abs(f - f_ss) <= tol_hz
    suffix_ok = np.logical_and.accumulate(within_band[::-1])[::-1]
    t_settle = t[np.argmax(suffix_ok)] if np.any(suffix_ok) else np.nan

    idx_r_min = np.argmin(rocof)
    idx_r_max = np.argmax(rocof)
    rocof_min = rocof[idx_r_min]
    rocof_max = rocof[idx_r_max]
    rocof_max_abs = np.max(np.abs(rocof))
    idx_r_abs = np.argmax(np.abs(rocof))

    metrics = {
        "f_ss": f_ss,
        "f_min": f_min,
        "t_min": t[idx_min],
        "f_max": f_max,
        "t_max": t[idx_max],
        "dev_down": dev_down,
        "dev_up": dev_up,
        "max_abs_dev": max_abs_dev,
        "overshoot_up": overshoot_up,
        "t_settle": t_settle,
        "rocof_min": rocof_min,
        "t_rocof_min": t[idx_r_min],
        "rocof_max": rocof_max,
        "t_rocof_max": t[idx_r_max],
        "rocof_max_abs": rocof_max_abs,
        "t_rocof_max_abs": t[idx_r_abs],
        "rocof_mean": np.mean(rocof),
        "rocof_rms": np.sqrt(np.mean(rocof**2)),
    }

    return metrics


def extract_ibr_peaks(plotter, iteration) -> Dict[str, float]:
    """Extract peak |Delta P| for each REGCV1 unit from plotter data."""
    idx = plotter.find("Pe REGCV1", idx_only=True)
    if not idx:
        return {}
    p_mat = np.asarray(plotter.get_values(idx)).transpose()
    peaks: Dict[str, float] = {}
    skip = iteration * SKIP_STEPS_PER_ITER
    for i, series in enumerate(p_mat):
        baseline = float(series[0])
        delta = series - baseline
        delta = delta[skip:]
        peak_max = float(np.max(delta))
        peak_min = float(np.min(delta))
        peak = peak_max if np.abs(peak_max) > np.abs(peak_min) else peak_min
        peaks[f"Delta_P_IBR_{i + 1}"] = peak
    return peaks


def build_feature_row(
    *,
    base_load_scale: float,
    load_step_scale: float,
    pq_names: Sequence[str],
    pq_p_before: np.ndarray,
    pq_q_before: np.ndarray,
    pq_p_after: np.ndarray,
    pq_q_after: np.ndarray,
    M_vec: Sequence[float],
    D_vec: Sequence[float],
    M_agg: float,
    D_agg: float,
) -> Dict[str, float]:
    """Assemble feature dictionary from inputs and PQ deltas."""
    features: Dict[str, float] = {
        "base_load_scale": float(base_load_scale),
        "load_step_scale": float(load_step_scale),
        "DELTA_PQ_tot": 0.0,
        "M_agg": float(M_agg),
        "D_agg": float(D_agg),
        "base_load_p_total": float(np.sum(pq_p_before)) if pq_p_before.size else 0.0,
        "base_load_q_total": float(np.sum(pq_q_before)) if pq_q_before.size else 0.0,
    }

    for i, (m_val, d_val) in enumerate(zip(M_vec, D_vec), start=1):
        features[f"M_{i}"] = float(m_val)
        features[f"D_{i}"] = float(d_val)

    delta_p_total = 0.0
    delta_q_total = 0.0
    for name, p_before, p_after, q_before, q_after in zip(
        pq_names, pq_p_before, pq_p_after, pq_q_before, pq_q_after
    ):
        dp = float(p_after - p_before)
        dq = float(q_after - q_before)
        features[f"DELTA_P_{name}"] = dp
        features[f"DELTA_Q_{name}"] = dq
        delta_p_total += dp
        delta_q_total += dq

    features["DELTA_PQ_tot"] = float(delta_p_total + delta_q_total)
    return features


def extract_simulation_row(
    *,
    ss,
    base_load_scale: float,
    load_step_scale: float,
    pq_names: Sequence[str],
    pq_p_before: np.ndarray,
    pq_q_before: np.ndarray,
    pq_p_after: np.ndarray,
    pq_q_after: np.ndarray,
    M_vec: Sequence[float],
    D_vec: Sequence[float],
    success: bool,
    genrou_M: np.ndarray,
    genrou_D: np.ndarray,
    regcv1_sn: np.ndarray,
    iteration: int,
) -> Dict[str, float]:
    """Build a row containing features, labels, and metadata for one simulation."""

    M_agg, D_agg = compute_md_agg(genrou_M, genrou_D, regcv1_sn, M_vec, D_vec)
    features = build_feature_row(
        base_load_scale=base_load_scale,
        load_step_scale=load_step_scale,
        pq_names=pq_names,
        pq_p_before=pq_p_before,
        pq_q_before=pq_q_before,
        pq_p_after=pq_p_after,
        pq_q_after=pq_q_after,
        M_vec=M_vec,
        D_vec=D_vec,
        M_agg=M_agg,
        D_agg=D_agg,
    )

    labels: Dict[str, float] = {name: np.nan for name in TARGET_ORDER}
    time_of_max_dev = np.nan

    if success:
        ss.TDS.load_plotter()
        plotter = ss.TDS.plotter
        time = np.asarray(plotter.get_values(0), dtype=float).reshape(-1)
        idx = plotter.find("omega COI", idx_only=True)
        if idx:
            f_coi = np.asarray(plotter.get_values(idx), dtype=float).reshape(-1) * SYSTEM_FREQ
            r_coi = np.gradient(f_coi, float(ss.TDS.config.tstep), axis=0)
            f0 = getattr(getattr(ss, "config", None), "freq", SYSTEM_FREQ) or SYSTEM_FREQ
            skip = iteration * SKIP_STEPS_PER_ITER
            metrics = compute_freq_metrics(time[skip:], f=f_coi[skip:], f0=f0, r=r_coi[skip:])

            dev_down = metrics.get("dev_down", np.nan)
            dev_up = metrics.get("dev_up", np.nan)
            if np.isfinite(dev_down) and np.isfinite(dev_up):
                if dev_down >= dev_up:
                    time_of_max_dev = float(metrics.get("t_min", np.nan))
                else:
                    time_of_max_dev = float(metrics.get("t_max", np.nan))

            for metric_key, label_name in FREQ_LABEL_MAP.items():
                if label_name in labels:
                    labels[label_name] = float(metrics.get(metric_key, np.nan))

            for key, val in extract_ibr_peaks(plotter, iteration).items():
                if key in labels:
                    labels[key] = float(val)

    row: Dict[str, float] = {}
    row.update(features)
    row.update(labels)
    row["time_max_dev"] = float(time_of_max_dev) if np.isfinite(time_of_max_dev) else np.nan
    row["success"] = bool(success)

    return row


In [ ]:
def predict_for_md(
    model,
    x_scaler,
    y_scaler,
    device,
    features,
    M_vec_cand,
    D_vec_cand,
    *,
    genrou_M=None,
    genrou_D=None,
    regcv1_sn=None,
) -> Dict[str, float]:
    feat_features = dict(features)
    if genrou_M is not None and genrou_D is not None and regcv1_sn is not None:
        M_agg, D_agg = compute_md_agg(genrou_M, genrou_D, regcv1_sn, M_vec_cand, D_vec_cand)
        feat_features["M_agg"] = M_agg
        feat_features["D_agg"] = D_agg

    feat_vec = build_feature_vector_for_md(feat_features, M_vec_cand, D_vec_cand)
    feat_norm = x_scaler.transform(feat_vec)
    with torch.no_grad():
        pred_norm = model(torch.from_numpy(feat_norm).to(device)).cpu().numpy()
    pred = y_scaler.inverse_transform(pred_norm).flatten()
    return {k: float(v) for k, v in zip(TARGET_ORDER, pred)}


def compute_best_md(
    current_M,
    current_D,
    features,
    *,
    model,
    x_scaler,
    y_scaler,
    device,
    rng,
    n_ibr,
    freq_limit,
    penalty_weight,
    M_bounds,
    D_bounds,
    n_candidates,
    genrou_M,
    genrou_D,
    regcv1_sn,
):
    base_M = np.asarray(current_M, dtype=float)
    base_D = np.asarray(current_D, dtype=float)

    price = PriceCoefficients(
        inertia=np.full(n_ibr, 0.0),
        damping=np.full(n_ibr, 0.0),
        delta_p=np.full(n_ibr, 1.0),
    )

    best = {
        "score": float("inf"),
        "cost": float("inf"),
        "feasible": False,
        "M": base_M,
        "D": base_D,
        "pred": {},
        "freq_metric": float("nan"),
    }

    for M_cand, D_cand in _candidate_generator(
        base_M=base_M,
        base_D=base_D,
        M_bounds=M_bounds,
        D_bounds=D_bounds,
        n_candidates=n_candidates,
        rng=rng,
    ):
        pred_map = predict_for_md(
            model,
            x_scaler,
            y_scaler,
            device,
            features,
            M_cand,
            D_cand,
            genrou_M=genrou_M,
            genrou_D=genrou_D,
            regcv1_sn=regcv1_sn,
        )

        delta_p = np.array(
            [pred_map.get(name, 0.0) for name in IBR_LABELS],
            dtype=float,
        )
        if delta_p.size < n_ibr:
            delta_p = np.pad(delta_p, (0, n_ibr - delta_p.size), mode="constant")

        cost = compute_dispatch_cost(M_cand, D_cand, delta_p, price)

        freq_vals = []
        for label in FREQ_METRIC_LABELS:
            val = pred_map.get(label)
            if val is None:
                continue
            freq_vals.append(abs(float(val)))
        freq_metric = max(freq_vals) if freq_vals else float("inf")

        feasible = freq_metric <= freq_limit
        penalty = penalty_weight * max(0.0, freq_metric - freq_limit)
        score = cost + penalty

        if score < best["score"]:
            best.update(
                score=float(score),
                cost=float(cost),
                feasible=bool(feasible),
                M=np.asarray(M_cand, dtype=float).copy(),
                D=np.asarray(D_cand, dtype=float).copy(),
                pred=pred_map,
                freq_metric=freq_metric,
            )

    return best


def update_current_md(best, current_M, current_D):
    if best.get("pred"):
        return np.asarray(best["M"], dtype=float), np.asarray(best["D"], dtype=float)
    return np.asarray(current_M, dtype=float), np.asarray(current_D, dtype=float)


In [ ]:
def fast_plot_bus_frequency(ss, save_path=None, max_points=2000, max_traces=None):
    """Faster plot: pulls all BusROCOF frequency traces in one call."""
    ss.TDS.load_plotter()
    plotter = ss.TDS.plotter
    f_nom = float(ss.config.freq)

    yidx = plotter.find("f BusROCOF", idx_only=True)
    if not yidx:
        raise RuntimeError("No BusROCOF frequency traces found. Did you add measurement devices?")

    t = np.asarray(plotter.get_values(0), dtype=float).reshape(-1)
    f_mat = np.asarray(plotter.get_values(yidx), dtype=float)

    if f_mat.ndim != 2:
        raise RuntimeError(f"Expected 2D frequency matrix, got shape {f_mat.shape}.")
    if f_mat.shape[0] == len(yidx):
        f_mat = f_mat.T

    f_mat = f_mat * f_nom

    if max_traces is not None and f_mat.shape[1] > max_traces:
        f_mat = f_mat[:, :max_traces]

    if max_points is not None and t.size > max_points:
        idx = np.linspace(0, t.size - 1, max_points, dtype=int)
        t = t[idx]
        f_mat = f_mat[idx, :]

    rocof = np.gradient(f_mat, t, axis=0)

    fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    axs[0].plot(t, f_mat, lw=0.8, alpha=0.7)
    axs[0].axhline(f_nom, ls="--", color="k", alpha=0.5)
    axs[0].set_ylabel("Frequency [Hz]")
    axs[0].set_title(f"{ss.name} - Bus Frequencies (fast)")

    axs[1].plot(t, rocof, lw=0.8, alpha=0.7)
    axs[1].set_xlabel("Time [s]")
    axs[1].set_ylabel("ROCOF [Hz/s]")
    axs[1].set_title("Bus ROCOF (fast)")

    for ax in axs:
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


In [ ]:
with open(csv_path, "w", newline="", encoding="utf-8") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
    writer.writeheader()

    total_steps = len(load_profile_df)
    if N_ITERS is not None:
        total_steps = min(int(N_ITERS), total_steps)

    pq_p_before = np.asarray(ss.PQ.p0.v, dtype=float).copy()
    pq_q_before = np.asarray(ss.PQ.q0.v, dtype=float).copy()

    for j in range(total_steps):
        base_load_scale = float(BASE_LOAD_SCALE)
        load_scale = apply_load_scale(ss, pq_base_p, pq_base_q, j)
        pq_p_after = np.asarray(ss.PQ.p0.v, dtype=float).copy()
        pq_q_after = np.asarray(ss.PQ.q0.v, dtype=float).copy()
        print("load_scale: ", j, load_scale)

        M_agg, D_agg = compute_md_agg(genrou_M, genrou_D, regcv1_sn, current_M, current_D)

        features = build_feature_row(
            base_load_scale=base_load_scale,
            load_step_scale=load_scale,
            pq_names=pq_names,
            pq_p_before=pq_p_before,
            pq_q_before=pq_q_before,
            pq_p_after=pq_p_after,
            pq_q_after=pq_q_after,
            M_vec=current_M,
            D_vec=current_D,
            M_agg=M_agg,
            D_agg=D_agg,
        )

        best = compute_best_md(
            current_M,
            current_D,
            features,
            model=model,
            x_scaler=x_scaler,
            y_scaler=y_scaler,
            device=device,
            rng=rng,
            n_ibr=N_IBR,
            freq_limit=FREQ_LIMIT,
            penalty_weight=PENALTY_WEIGHT,
            M_bounds=M_BOUNDS,
            D_bounds=D_BOUNDS,
            n_candidates=N_CANDIDATES,
            genrou_M=genrou_M,
            genrou_D=genrou_D,
            regcv1_sn=regcv1_sn,
        )

        current_M, current_D = update_current_md(best, current_M, current_D)
        update_regcv1_params(ss, current_M, current_D)

        ss.TDS.config.tf = float(STEP_DURATION * (j + 1))
        print(ss.TDS.config.tf)

        success = bool(ss.TDS.run())

        row = extract_simulation_row(
            ss=ss,
            base_load_scale=base_load_scale,
            load_step_scale=load_scale,
            pq_names=pq_names,
            pq_p_before=pq_p_before,
            pq_q_before=pq_q_before,
            pq_p_after=pq_p_after,
            pq_q_after=pq_q_after,
            M_vec=current_M,
            D_vec=current_D,
            success=success,
            genrou_M=genrou_M,
            genrou_D=genrou_D,
            regcv1_sn=regcv1_sn,
            iteration=j,
        )

        labels = {k: row.get(k, float("nan")) for k in TARGET_ORDER}

        if success:
            errors = []
            print()
            print(f"=== NN Prediction vs. Sim labels (load_scale={load_scale:.4f}) at iter {j} ===")
            for name in TARGET_ORDER:
                p = float(best["pred"].get(name, float("nan")))
                target_val = labels.get(name, None)
                if target_val is None:
                    print(f"{name:20s}: pred={p:.4f} (label missing)")
                else:
                    true_val = float(target_val)
                    errors.append(p - true_val)
                    print(f"{name:20s}: pred={p:.4f} | true={true_val:.4f}")

            if errors:
                rmse = float(np.sqrt(np.mean(np.square(errors))))
                print()
                print(f"RMSE (available targets): {rmse:.4f}")

            print()
            print(f"TDS success flag for iter {j}: {success}")

        freq_key = "max(|" + ", |".join(FREQ_METRIC_LABELS) + "|)" if FREQ_METRIC_LABELS else "freq_metric"
        print()
        print("=== Optimal virtual inertia (NN-based) ===")
        print(f"Selected M: {best['M'].tolist()}")
        print(f"Selected D: {best['D'].tolist()}")
        print(
            f"Predicted {freq_key}: {best['freq_metric']:.4f} Hz "
            f"(limit={FREQ_LIMIT:.4f}) feasible={best['feasible']}"
        )
        print(f"Dispatch cost (weights): {best['cost']:.4f}  | score={best['score']:.4f}")

        log_lines = [
            f"iter={j} load_scale={load_scale:.4f}",
            f"success={bool(success)} feasible={bool(best['feasible'])} "
            f"cost={best['cost']:.4f} score={best['score']:.4f} freq_metric={best['freq_metric']:.4f}",
            f"M={best['M'].tolist()}",
            f"D={best['D'].tolist()}",
            "pred_vs_true:",
        ]
        row_out = {
            "iter": j,
            "load_scale": float(load_scale),
            "success": bool(success),
            "feasible": bool(best['feasible']),
            "cost": float(best['cost']),
            "score": float(best['score']),
            "freq_metric": float(best['freq_metric']),
        }
        for i_m, m_val in enumerate(best['M'][:N_IBR], start=1):
            row_out[f"M_{i_m}"] = float(m_val)
        for i_d, d_val in enumerate(best['D'][:N_IBR], start=1):
            row_out[f"D_{i_d}"] = float(d_val)

        for name in TARGET_ORDER:
            pred_val = float(best['pred'].get(name, float("nan")))
            true_raw = labels.get(name, float("nan"))
            try:
                true_val = float(true_raw)
            except (TypeError, ValueError):
                true_val = float("nan")
            log_lines.append(f"  {name}: pred={pred_val:.4f} | true={true_val:.4f}")
            row_out[f"pred_{name}"] = pred_val
            row_out[f"true_{name}"] = true_val

        with open(text_log_path, "a", encoding="utf-8") as text_file:
            text_file.write(" ".join(log_lines))
            text_file.write("\n\n")

        writer.writerow(row_out)


In [ ]:
df = pd.read_csv(csv_path).head(40)
iter_vals = df["iter"].values

error_targets = [t for t in TARGET_ORDER if f"pred_{t}" in df.columns and f"true_{t}" in df.columns]
if error_targets:
    err_stack = []
    for t in error_targets:
        pred = df[f"pred_{t}"].to_numpy(dtype=float)
        true = df[f"true_{t}"].to_numpy(dtype=float)
        err_stack.append(np.abs(pred - true))
    mean_abs_error = np.nanmean(np.vstack(err_stack), axis=0)
else:
    mean_abs_error = np.full_like(iter_vals, np.nan, dtype=float)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

axes[0].step(iter_vals, df["load_scale"], label="load_scale", where="post")
axes[0].set_title("Load Scale")
axes[0].set_xlabel("iter")
axes[0].set_ylabel("load_scale")
axes[0].grid(True, alpha=0.3)

axes[1].step(iter_vals, df["cost"], label="cost", where="post")
axes[1].set_title("Cost")
axes[1].set_xlabel("iter")
axes[1].set_ylabel("cost")
axes[1].grid(True, alpha=0.3)

axes[2].step(iter_vals, mean_abs_error, label="mean_abs_error", where="post")
axes[2].set_title("Mean Abs Error (pred vs true)")
axes[2].set_xlabel("iter")
axes[2].set_ylabel("error")
axes[2].grid(True, alpha=0.3)

if PLOT_TARGETS is None:
    plot_targets = TARGET_ORDER[:3]
else:
    plot_targets = list(PLOT_TARGETS)

for t in plot_targets:
    pred_col = f"pred_{t}"
    true_col = f"true_{t}"
    if pred_col in df.columns and true_col in df.columns:
        axes[3].step(iter_vals, df[pred_col], label=f"pred_{t}", where="post")
        axes[3].step(iter_vals, df[true_col], linestyle="--", label=f"true_{t}", where="post")

axes[3].set_title("Predicted vs True")
axes[3].set_xlabel("iter")
axes[3].set_ylabel("value")
axes[3].grid(True, alpha=0.3)
axes[3].legend(fontsize=8, ncol=2)

plt.tight_layout()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for i in range(1, N_IBR + 1):
    m_col = f"M_{i}"
    d_col = f"D_{i}"
    if m_col in df.columns:
        axes[0].plot(iter_vals, df[m_col], marker="o", label=m_col)
    if d_col in df.columns:
        axes[1].plot(iter_vals, df[d_col], marker="o", label=d_col)

axes[0].set_title("M values")
axes[0].set_ylabel("M")
axes[0].grid(True, alpha=0.3)
axes[0].legend(ncol=2, fontsize=8)

axes[1].set_title("D values")
axes[1].set_xlabel("iter")
axes[1].set_ylabel("D")
axes[1].grid(True, alpha=0.3)
axes[1].legend(ncol=2, fontsize=8)

plt.tight_layout()


In [ ]:
fixed_baseline_csv = ""
fixed_baseline_df = None
if fixed_baseline_csv:
    fixed_baseline_df = pd.read_csv(fixed_baseline_csv).sort_values("iter").reset_index(drop=True)


In [ ]:
df = pd.read_csv(csv_path).sort_values("iter").reset_index(drop=True)
if df.empty:
    raise RuntimeError("vis_rted_tds_results.csv is empty. Run the simulation first.")

def _get_col(frame, name):
    return frame[name].to_numpy(dtype=float) if frame is not None and name in frame.columns else None

def _metrics_from_frame(frame):
    freq_metric = None
    if FREQ_METRIC_LABELS:
        vals = []
        for label in FREQ_METRIC_LABELS:
            col = f"true_{label}"
            if col in frame.columns:
                vals.append(np.abs(frame[col].to_numpy(dtype=float)))
        if vals:
            freq_metric = np.maximum.reduce(vals)

    rocof_metric = None
    if ROCOF_LABELS:
        vals = []
        for label in ROCOF_LABELS:
            col = f"true_{label}"
            if col in frame.columns:
                vals.append(np.abs(frame[col].to_numpy(dtype=float)))
        if vals:
            rocof_metric = np.maximum.reduce(vals)

    cost = _get_col(frame, "cost")
    return freq_metric, rocof_metric, cost

freq_metric, rocof_metric, cost = _metrics_from_frame(df)

baseline_label = "iter-0 (no-VIS)"
baseline_iter0 = {
    "freq_metric": float(freq_metric[0]) if freq_metric is not None else float("nan"),
    "rocof_metric": float(rocof_metric[0]) if rocof_metric is not None else float("nan"),
    "cost": float(cost[0]) if cost is not None else float("nan"),
}

fixed_label = "fixed M/D baseline"
fixed_metrics = None
if fixed_baseline_df is not None and not fixed_baseline_df.empty:
    f_freq, f_rocof, f_cost = _metrics_from_frame(fixed_baseline_df)
    fixed_metrics = {
        "freq_metric": float(f_freq[0]) if f_freq is not None else float("nan"),
        "rocof_metric": float(f_rocof[0]) if f_rocof is not None else float("nan"),
        "cost": float(f_cost[0]) if f_cost is not None else float("nan"),
    }

fig, axs = plt.subplots(2, 2, figsize=(12, 8))
axs = axs.ravel()

if freq_metric is not None:
    axs[0].plot(df["iter"], freq_metric, marker="o", label="VIS")
    axs[0].axhline(baseline_iter0["freq_metric"], color="k", ls="--", alpha=0.5, label=baseline_label)
    if fixed_metrics is not None:
        axs[0].axhline(fixed_metrics["freq_metric"], color="tab:green", ls=":", alpha=0.8, label=fixed_label)
    axs[0].set_title("Max frequency deviation")
    axs[0].set_ylabel("Hz")
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)

if rocof_metric is not None:
    axs[1].plot(df["iter"], rocof_metric, marker="o", color="tab:orange", label="VIS")
    axs[1].axhline(baseline_iter0["rocof_metric"], color="k", ls="--", alpha=0.5, label=baseline_label)
    if fixed_metrics is not None:
        axs[1].axhline(fixed_metrics["rocof_metric"], color="tab:green", ls=":", alpha=0.8, label=fixed_label)
    axs[1].set_title("Max |RoCoF|")
    axs[1].set_ylabel("Hz/s")
    axs[1].legend()
    axs[1].grid(True, alpha=0.3)

if cost is not None:
    axs[2].plot(df["iter"], cost, marker="o", color="tab:blue", label="VIS cost")
    axs[2].axhline(baseline_iter0["cost"], color="k", ls="--", alpha=0.5, label=baseline_label)
    if fixed_metrics is not None:
        axs[2].axhline(fixed_metrics["cost"], color="tab:green", ls=":", alpha=0.8, label=fixed_label)
    axs[2].set_title("Dispatch cost")
    axs[2].set_xlabel("iter")
    axs[2].set_ylabel("cost")
    axs[2].legend()
    axs[2].grid(True, alpha=0.3)

if freq_metric is not None and cost is not None:
    axs[3].scatter(cost, freq_metric, c=df["iter"], cmap="viridis", s=50)
    axs[3].set_title("Cost vs frequency deviation")
    axs[3].set_xlabel("cost")
    axs[3].set_ylabel("Hz")
    axs[3].grid(True, alpha=0.3)

plt.tight_layout()

summary = []
summary.append({
    "case": baseline_label,
    **baseline_iter0,
})
if fixed_metrics is not None:
    summary.append({
        "case": fixed_label,
        **fixed_metrics,
    })

best_idx = int(df["iter"].iloc[np.nanargmin(freq_metric)]) if freq_metric is not None else int(df["iter"].iloc[-1])
best_row = df[df["iter"] == best_idx].iloc[0]
summary.append({
    "case": f"best VIS (iter={best_idx})",
    "freq_metric": float(freq_metric[df["iter"] == best_idx][0]) if freq_metric is not None else float("nan"),
    "rocof_metric": float(rocof_metric[df["iter"] == best_idx][0]) if rocof_metric is not None else float("nan"),
    "cost": float(best_row["cost"]) if "cost" in best_row else float("nan"),
})

pd.DataFrame(summary)


In [ ]:
df = pd.read_csv(csv_path).sort_values("iter").reset_index(drop=True)
if df.empty:
    raise RuntimeError("vis_rted_tds_results.csv is empty. Run the simulation first.")

load_scale = df["load_scale"].to_numpy(dtype=float) if "load_scale" in df.columns else np.arange(len(df))

m_cols = [c for c in df.columns if c.startswith("M_") and c[2:].isdigit()]
d_cols = [c for c in df.columns if c.startswith("D_") and c[2:].isdigit()]

if m_cols and d_cols:
    m_agg = df[m_cols].mean(axis=1)
    d_agg = df[d_cols].mean(axis=1)

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.scatter(load_scale, m_agg, marker="o", label="M_agg")
    ax.scatter(load_scale, d_agg, marker="o", label="D_agg")
    ax.set_title("Aggregated M/D vs load scale")
    ax.set_xlabel("load_scale")
    ax.set_ylabel("avg M/D")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()

err_cols = []
for label in FREQ_LABEL_MAP.values():
    err_cols.append((label, f"pred_{label}", f"true_{label}"))
for label in IBR_LABELS:
    err_cols.append((label, f"pred_{label}", f"true_{label}"))

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
for label, pcol, tcol in err_cols:
    if pcol in df.columns and tcol in df.columns:
        err = np.abs(df[pcol].to_numpy(dtype=float) - df[tcol].to_numpy(dtype=float))
        ax.scatter(load_scale, err, marker="o", label=label)
ax.set_title("Absolute error: metrics vs load scale")
ax.set_xlabel("load_scale")
ax.set_ylabel("abs error")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()


In [ ]:
df = pd.read_csv(csv_path).sort_values("iter").reset_index(drop=True)
if df.empty:
    raise RuntimeError("vis_rted_tds_results.csv is empty. Run the simulation first.")

x = df["iter"].to_numpy(dtype=float) if "iter" in df.columns else np.arange(len(df))
load_scale = df["load_scale"].to_numpy(dtype=float) if "load_scale" in df.columns else None

metrics = []
for col in df.columns:
    if col.startswith("pred_"):
        name = col[len("pred_"):]
        true_col = f"true_{name}"
        if true_col in df.columns:
            metrics.append((name, col, true_col))

if not metrics:
    raise RuntimeError("No pred_/true_ metric columns found in the CSV.")

max_panels = 8
metrics = metrics[:max_panels]

fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharex=True)
axes = axes.ravel()

for ax, (name, pred_col, true_col) in zip(axes, metrics):
    pred = df[pred_col].to_numpy(dtype=float)
    true = df[true_col].to_numpy(dtype=float)
    err = np.abs(pred - true)
    ax.step(x, err, where="mid", label="abs error")
    ax.set_title(name)
    ax.set_ylabel("error")
    ax.grid(True, alpha=0.3)
    if load_scale is not None:
        ax2 = ax.twinx()
        ax2.plot(x, load_scale, color="tab:gray", alpha=0.4, label="load_scale")
        ax2.set_ylabel("load_scale")

for ax in axes[len(metrics):]:
    ax.axis("off")

for ax in axes[-4:]:
    ax.set_xlabel("iter")

fig.suptitle("Absolute error per metric (step) with load scale", y=1.02)
plt.tight_layout()
